# 081 — Serving, batching y cachés

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** KV/token = 2·40·40·128·2 = **819 200 B ≈ 0,82 MB**.
A 8 192 tokens: 0,82 MB × 8 192 ≈ **6,7 GB** por secuencia.
Con GQA-8: 2·40·8·128·2 = 163 840 B ≈ 0,164 MB/token → **1,34 GB** (5× menos).

**Ejercicio 2.** Disponible: 48 GB. Estática a 8 192: 1,34 GB/slot → **35
secuencias**. Paginada a 2 048 reales: 0,164 MB × 2 048 ≈ 0,336 GB → **~143
secuencias** (~4× más): exactamente la ganancia que reporta vLLM al eliminar la
reserva por el máximo.

**Ejercicio 3.** El lote dura 800 pasos × 4 slots = 3 200 pasos-slot.
Útiles: 100+200+400+800 = 1 500. Desperdiciados: 1 700 → utilización **46,9 %**.
Con continuous batching los slots liberados sirven otras requests: utilización
cercana al 100 % y ~2,1× más throughput en este ejemplo.

**Ejercicio 4.** Sin métricas por fase (TTFT/TPOT, ocupación de bloques, colas) es
imposible saber qué palanca tocar; el laboratorio de observabilidad modela ese
contrato de medición.

In [ ]:
# Ejercicio 1
kv_tok = lambda capas, kvh, dh, b: 2 * capas * kvh * dh * b
full = kv_tok(40, 40, 128, 2)
gqa = kv_tok(40, 8, 128, 2)
print(f"full={full} B/token  cache8192={full*8192/1e9:.2f} GB  gqa={gqa} B/token")

# Ejercicio 2
disp = (80 - 28 - 4) * 1e9
print("estatico:", int(disp // (gqa * 8192)), "paginado:", int(disp // (gqa * 2048)))

# Ejercicio 3
longs = [100, 200, 400, 800]
total = max(longs) * len(longs)
utiles = sum(longs)
print(f"desperdicio={total-utiles}  utilizacion={utiles/total:.1%}")  # 46.9%

# Ejercicio 4
result = run_lab("observability", seed=81)
assert result["kind"] == "observability"
assert result["evidence"] and result["limitations"]
show(result)

## Reflexión

1. ¿Por qué el decode está limitado por ancho de banda y no por FLOPs, y qué
   implica para elegir GPU de serving frente a GPU de entrenamiento?
2. Un prompt de sistema de 2 000 tokens compartido por todos los usuarios: ¿qué
   mecanismo de esta clase lo convierte en casi gratis y por qué?
3. Si tu p99 de TPOT empeora al subir el tamaño máximo de lote, ¿qué compromiso
   estás viendo y cómo fijarías el límite?